# Phase 4 — 교차모델 behavior 주입 (성격 착시 재현)

**질문:** v_behavior를 mid-layer에 주입하면 **자유생성 행동은 움직이나 자기보고 digit은 3에 고정**되는 성격 착시가 **Mistral·Qwen에도** 나타나나? (Gemma에서만 확인됨: behavior_proj −17.6→+20.7, 자기보고 `dE≈0.017`.)

- **착시 재현** = 행동 움직이는데 `dE_ft`≈0 → 현상이 모델 불문(강건).
- **착시 약함** = 주입이 `dE_ft`도 움직임 → 그 모델은 행동·자기보고가 더 통합적(=최신모델 일치 가설 지지).
- 이는 이중해리의 **첫 번째 팔(behavior 축)**의 교차모델 재현. 완전한 이중해리(v_selfreport)는 Phase 1 별도.

**파이프라인(모델별 순차):** `extract_activations` → `build_vectors`(v_behavior) → `steer_eval`(α-sweep). 각 sweep 행에 `behavior_proj` + 자기보고 `E_ft` 동시 기록.

**실행 순서:** GPU → 의존성 → HF 로그인 → 번들(`steering_crossmodel_bundle.zip`) 업로드 → 모델 설정 → 스모크 → 전체 → 비교표.

> **런타임:** 3모델 full extract는 무거움(각 ~2,976 forward). 무료 T4면 `STEER_4BIT=1`(설정 셀). **-it/Instruct 전용.**

## 1. GPU + 의존성 + HF 로그인

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q "transformers>=4.45" accelerate huggingface_hub bitsandbytes sentencepiece python-dotenv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # gemma-2-9b-it 라이선스 수락 필요. Mistral/Qwen(Apache)은 로그인 없어도 됨.

## 2. 번들 업로드

`steering_crossmodel_bundle.zip` 선택 — `common.py` + `steering/{extract_activations,build_vectors,diagnostics,steer_eval,gemma_common}.py` + `data/facets_en.json` + `outputs/pairs_final.jsonl`.
**주의:** Qwen no-BOS 수정이 반영된 최신 `gemma_common.py`가 들어있어야 함(번들 재빌드 후 업로드).

In [ ]:
from google.colab import files
import zipfile, os, subprocess
print('steering_crossmodel_bundle.zip 를 선택하세요 ...')
up = files.upload()
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/asteer')
%cd /content/asteer
print(subprocess.run(['find','.','-maxdepth','2','-type','f'], capture_output=True, text=True).stdout)

## 3. 모델 설정 (여기만 바꾸면 됨)

Gemma는 기존 착시 재현 위해 layer 20 고정. Mistral/Qwen은 auto mid-layer(각 15/13). **주입은 층에 민감** → 진지한 결과엔 모델별 `--layer-sweep`으로 스위트스팟 재탐색 권장.

In [ ]:
import os
# 무료 T4(16GB)면 '1'(4-bit) 권장. Colab Pro L4/A100면 '0'(추출 품질↑).
os.environ['STEER_4BIT'] = '1'
MODELS = [
    {'slug': 'gemma9b',   'id': 'google/gemma-2-9b-it',               'layer': '20'},   # 앵커(착시 재현)
    {'slug': 'mistral7b', 'id': 'mistralai/Mistral-7B-Instruct-v0.3', 'layer': None},   # auto mid-layer(~15)
    {'slug': 'qwen7b',    'id': 'Qwen/Qwen2.5-7B-Instruct',           'layer': None},   # auto mid-layer(~13)
]
print('STEER_4BIT =', os.environ['STEER_4BIT'])
for m in MODELS: print(' -', m['id'], '| layer', m['layer'] or 'auto')

## 4. 스모크 (모델별: extract `--limit 32` → build → steer_eval `--smoke`)

모델 로딩·추출·벡터·스윕이 크래시 없이 도는지 + Qwen 추출 확인. build 메타의 `cos_V1_V2`가 sane한지도 로그로 확인.

In [ ]:
# 순차 실행: 기본 dir(artifacts/activations, artifacts/vectors) 재사용 → 다음 모델이 덮어씀(무방, 결과 JSON은 per-model 저장).
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n########## SMOKE {m['id']} ##########")
    !python steering/extract_activations.py --model {m['id']} --limit 32
    !python steering/build_vectors.py {lay}
    !python steering/steer_eval.py --model {m['id']} {lay} --smoke --out artifacts/vectors/steer_{m['slug']}_smoke.json

## 5. 전체 실행 (extract → build → steer_eval)

In [ ]:
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n########## FULL {m['id']} ##########")
    !python steering/extract_activations.py --model {m['id']}
    !python steering/build_vectors.py {lay}
    !python steering/steer_eval.py --model {m['id']} {lay} --out artifacts/vectors/steer_{m['slug']}.json

## 6. 교차모델 비교표 + 다운로드

In [ ]:
import json, os
from google.colab import files
hdr = f"{'model':10} {'L':>3} {'R':>6} {'bproj(min->max)':>17} {'Bcorr':>6} {'dE_ft':>7} {'dN_ft':>7}  illusion?"
print(hdr); print('-' * len(hdr))
for m in MODELS:
    p = f"artifacts/vectors/steer_{m['slug']}.json"
    try:
        r = json.load(open(p))
    except FileNotFoundError:
        print(f"{m['slug']:10}  (missing {p})"); continue
    bp = [s['behavior_proj'] for s in r['sweep']]
    sr = r['summary']['self_report'].get('first_token_isolated', {})
    dE, dN = sr.get('dE'), sr.get('dN')
    if dE is None:
        print(f"{m['slug']:10}  (no self_report)"); continue
    Bcorr = r['summary'].get('B_behavior_dose_corr')
    rng = max(bp) - min(bp)
    verdict = 'REPRODUCED (behav O / self X)' if (rng > 2 and abs(dE) < 0.5) else \
              ('injection moves digit too' if abs(dE) >= 0.5 else 'inconclusive')
    print(f"{m['slug']:10} {r['layer']:>3} {r['R']:>6.1f} {min(bp):+.1f}->{max(bp):+.1f}".ljust(45) +
          f" {Bcorr:>6.2f} {dE:>7.3f} {dN:>7.3f}  {verdict}")
print('\n해석: behavior_proj 넓게 & dE_ft(iso)≈0 → 성격 착시 재현(행동만 움직이고 자기보고 고정).')
print('      dE_ft 큼 → 주입이 자기보고도 움직임 = 그 모델은 더 통합적(착시 약함).')
print('      ⚠️ behavior_proj는 주입벡터 자체 투영이라 "행동 움직임"은 부분적으로 설계상 당연 — 진짜 신호는 dE≈0.')
for m in MODELS:
    p = f"artifacts/vectors/steer_{m['slug']}.json"
    if os.path.exists(p): files.download(p)

---
### 해석 가이드 (성격 착시 = 이중해리 첫 팔)
- **착시 재현** = `behavior_proj`가 α 스윕에서 넓게 움직이는데 `dE_ft(iso)`≈0. Gemma 기준값 behavior_proj −17.6→+20.7, dE≈0.017.
- **착시 약함/없음** = 주입이 `dE_ft`도 크게 움직임 → 그 모델은 행동·자기보고가 더 통합적(사용자 가설 "최신모델 일치" 지지).

### 주의
- **behavior_proj는 주입벡터 자체에 투영**(`steer_eval.behavior`) → "행동이 움직였다"는 부분적으로 설계상 당연. 진짜 착시 신호는 **dE≈0(자기보고 고정)**. `sweep[*].sample` 생성텍스트로 실제 행동 변화 육안 확인(엄밀 측정=LLM 심판은 향후).
- **R·behavior_proj 절대값은 모델간 스케일 다름** → 패턴(움직임 vs dE≈0)으로 비교, 절대값 비교 금지.
- **층:** auto mid-layer 첫 패스. 주입은 층 민감 → 진지한 결과엔 모델별 `--layer-sweep` 스위트스팟 재탐색.
- **4-bit 추출:** T4 필수지만 활성화 degrade → 가능하면 L4/A100 bf16.

### 다음
- 완전한 이중해리는 v_selfreport(Phase 1)까지 필요 — 이 노트북은 **behavior 팔만**.